# Semantic Search using Sentence Embeddings

This notebook implements meaning-based retrieval
using dense vector embeddings and cosine similarity.

In [2]:
# Installing sentence-transformers

In [1]:
import sys
!{sys.executable} -m pip install sentence-transformers

  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.2.3-py3-none-any.whl (494 kB)


In [15]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
##Genres list

In [16]:
GENRES = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Drama", "Family", "Fantasy", "History", "Horror",
    "Music", "Mystery", "Romance", "Science Fiction",
    "Thriller", "War", "Western"
]

## Loading Processed Data

In [17]:
df = pd.read_csv("../data/processed/movies_semantic.csv")
df.shape

(4803, 5)

In [18]:
df['genres_clean'] = df['genres_clean'].fillna('')
df['semantic_text'] = df['semantic_text'].astype(str)

In [19]:
df.isnull().sum()

id               0
title            0
semantic_text    0
genres_clean     0
vote_average     0
dtype: int64

## Loading Pretrained Embedding Model

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


##### THis converts text into 384-dimensional dense vectors

## Generating Movie Embeddings

In [9]:
movie_embeddings = model.encode(
    df['semantic_text'].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/151 [00:00<?, ?it/s]

In [10]:
movie_embeddings.shape

(4803, 384)

## Saving Embeddings

In [11]:
np.save("../data/processed/movie_embeddings.npy", movie_embeddings)

## Building Semantic Search Function

In [20]:
def semantic_search(query, top_n=5):
    query_embedding = model.encode([query])
    
    similarities = cosine_similarity(query_embedding, movie_embeddings)
    
    top_indices = similarities[0].argsort()[-top_n:][::-1]
    
    results = df.iloc[top_indices][['title', 'vote_average']]
    
    return results

### detects genres

In [21]:
def detect_genres(query):
    detected = []
    for genre in GENRES:
        if genre.lower() in query.lower():
            detected.append(genre)
    return detected

## Smart Hybrid Search

In [22]:
def hybrid_search(query, top_n=5):
    detected_genres = detect_genres(query)
    
    if detected_genres:
        # Filter by detected genres
        mask = df['genres_clean'].apply(
            lambda x: any(g in x for g in detected_genres)
        )
        
        filtered_indices = df[mask].index
        filtered_embeddings = movie_embeddings[filtered_indices]
        
        query_embedding = model.encode([query])
        similarities = cosine_similarity(query_embedding, filtered_embeddings)
        
        top_indices = similarities[0].argsort()[-top_n:][::-1]
        selected_indices = filtered_indices[top_indices]
        
        return df.loc[selected_indices][['title', 'vote_average']]
    
    else:
        # No genre detected → normal semantic search
        return semantic_search(query, top_n)

In [22]:
semantic_search("loneliness in space")

,title,vote_average
4515,The Station Agent,7.4
239,Gravity,7.3
95,Interstellar,8.1
778,Meet Dave,5.0
3729,Cargo,5.9


In [23]:
semantic_search("science fiction space isolation survival")

,title,vote_average
461,Lost in Space,5.0
1198,Escape from Planet Earth,5.7
1531,Moonraker,5.9
56,Star Trek Beyond,6.6
239,Gravity,7.3


In [24]:
semantic_search(
    "a science fiction movie about astronauts isolated in deep space struggling to survive"
)

,title,vote_average
239,Gravity,7.3
1531,Moonraker,5.9
1198,Escape from Planet Earth,5.7
643,Space Cowboys,6.3
2757,The Astronaut Farmer,6.2


In [25]:
semantic_search("a science fiction movie about space mission and father-daughter relationship")

,title,vote_average
1473,The Astronaut's Wife,5.4
4190,Under the Same Moon,7.0
141,Mars Needs Moms,5.5
3355,Flirting with Disaster,6.3
95,Interstellar,8.1


In [26]:
semantic_search("a highly motivating boxing story")

,title,vote_average
1295,Against the Ropes,4.5
4331,Rocky,7.5
380,Cinderella Man,7.3
4617,Fighting Tommy Riley,5.3
2916,The Fighter,7.2


#### Duplicating tf-idf vectorization for comparing the results

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(df['semantic_text'])

def keyword_search(query, top_n=5):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix)
    top_indices = similarities[0].argsort()[-top_n:][::-1]
    return df.iloc[top_indices][['title', 'vote_average']]

# System Evaluation & Comparison

In [24]:
evaluation_queries = [
    "loneliness in space",
    "italian mafia family business",
    "emotional father daughter relationship",
    "highly motivating boxing story",
    "romantic high school comedy",
    "war survival movie",
    "psychological thriller with mystery",
    "underdog sports drama"
]

In [25]:
for query in evaluation_queries:
    print("\n==============================")
    print("Query:", query)
    
    print("\nTF-IDF:")
    print(keyword_search(query))
    
    print("\nSemantic:")
    print(semantic_search(query))
    
    print("\nHybrid:")
    print(hybrid_search(query))


Query: loneliness in space

TF-IDF:
               title  vote_average
1531       Moonraker           5.9
461    Lost in Space           5.0
2198         Lockout           5.8
4332  Silent Running           6.3
239          Gravity           7.3

Semantic:
                  title  vote_average
4518  The Station Agent           7.4
239             Gravity           7.3
95         Interstellar           8.1
778           Meet Dave           5.0
365             Contact           7.2

Hybrid:
                  title  vote_average
4518  The Station Agent           7.4
239             Gravity           7.3
95         Interstellar           8.1
778           Meet Dave           5.0
365             Contact           7.2

Query: italian mafia family business

TF-IDF:
                    title  vote_average
3536       Mambo Italiano           5.6
3337        The Godfather           8.4
1225     Mickey Blue Eyes           5.3
1466  Unfinished Business           5.0
3859      Lords of London     

# Comparative Evaluation

Observations:

1. TF-IDF performs well for exact keyword queries but struggles with conceptual similarity.
2. Semantic search captures thematic and emotional meaning effectively.
3. Hybrid search improves domain-specific queries by combining genre filtering with semantic ranking.
4. For multi-intent queries, hybrid provides the most stable results.